# 04 — Entrenamiento final y submit a Kaggle

**Entrada** : `{BUCKET}/exp/<EXPERIMENTO>/resultado.json` (lo deja `03_Optuna`)
**Salida**  : `submission.csv` + submit a Kaggle

Qué hace distinto de `03`:

`03_Optuna` entrena reservando meses para validar y testear — tiene que hacerlo,
porque necesita una estimación honesta del error. Acá ya no hay nada que estimar: los
hiperparámetros están elegidos, así que se entrena con **todos los meses
supervisados**, incluidos los que `03` había apartado. Más datos, mismo modelo.

Tres cosas más:

- **Ensemble de semillas**: entrena N modelos idénticos salvo la semilla y promedia.
  Baja la varianza sin tocar el sesgo, y es de lo más barato que hay.
- **Reconstrucción a toneladas**: igual que en `03`, según la variable respuesta que
  se haya usado.
- **Armado de la entrega**: los 780 productos de `product_id_apredecir201912.txt`,
  los que no tengan predicción van con 0.

## 0 — Ambiente

In [ ]:
!pip install -q uv
!uv pip install -q pyarrow polars lightgbm pandas kaggle

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

import numpy as np
import polars as pl
import pandas as pd
import lightgbm as lgb


def resolver_bucket() -> Path:
    # 1) LABO3_BUCKET: para correr fuera de la nube (server propio, notebook local).
    env = os.environ.get("LABO3_BUCKET")
    if env:
        p = Path(env).expanduser().resolve()
        p.mkdir(parents=True, exist_ok=True)
        return p
    # 2) rutas conocidas: Colab y la VM de GCP
    for cand in ("/content/buckets/b1", "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Opciones: "
        "(a) Colab / VM de GCP -> corre la celda de init del ambiente; "
        "(b) server propio o local -> defini LABO3_BUCKET antes de esta celda, ej. "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET   = resolver_bucket()
RUTA_FE  = BUCKET / "datasets_fe"
RUTA_EXP = BUCKET / "exp"
RUTA_RAW = BUCKET / "datasets"

print(f"BUCKET: {BUCKET}")
print(f"\nExperimentos disponibles en {RUTA_EXP}:")
for d in sorted(RUTA_EXP.iterdir()):
    if d.is_dir() and (d / "resultado.json").exists():
        r = json.load(open(d / "resultado.json", encoding="utf-8"))
        print(f"   wape_test={r.get('wape_test', float('nan')):.4f}   {d.name}")

## 1 — Palancas

`experimento` es el nombre exacto de la carpeta en `exp/`. Si lo dejás en `None`,
toma **el de menor `wape_test`** del leaderboard, que es la elección por defecto
razonable: el test es la única estimación no contaminada por la búsqueda de Optuna.

In [ ]:
PARAM = {
    # Carpeta del experimento en exp/. None = el de mejor wape_test.
    'experimento': None,

    # Semillas del ensemble: un modelo por semilla, se promedian las predicciones.
    # [102191] = un solo modelo. 5 semillas suele dar una mejora chica pero gratis.
    'semillas_ensemble': [102191, 202301, 314159, 777777, 987654],

    # Mes objetivo de la entrega. El pipe predice a t+horizonte; con datos hasta
    # 201912 y horizonte 2, el mes a entregar es 202002.
    'periodo_objetivo': 202002,

    # Kaggle
    'kaggle_competition': 'labo-iii-2026-rosario',
    'submit': True,                 # False = solo genera el CSV, no lo sube
    'mensaje_submit': None,         # None = se arma solo con las metricas

    # Piso de las predicciones: no existen ventas negativas.
    'clip_min': 0.0,
}
print(PARAM)

In [ ]:
# ── Elegir el experimento ────────────────────────────────────────────────
disponibles = [d for d in sorted(RUTA_EXP.iterdir())
               if d.is_dir() and (d / "resultado.json").exists()]
if not disponibles:
    raise RuntimeError(f"No hay experimentos con resultado.json en {RUTA_EXP}. Corre 03_Optuna.")

if PARAM['experimento'] is None:
    def _wt(d):
        r = json.load(open(d / "resultado.json", encoding="utf-8"))
        v = r.get("wape_test")
        return float(v) if v is not None else float("inf")
    DIR_EXP = min(disponibles, key=_wt)
    print(f"Sin experimento indicado -> se elige el de mejor wape_test")
else:
    DIR_EXP = RUTA_EXP / PARAM['experimento']
    if not (DIR_EXP / "resultado.json").exists():
        raise FileNotFoundError(
            f"No existe {DIR_EXP/'resultado.json'}.\nDisponibles: {[d.name for d in disponibles]}")

CFG = json.load(open(DIR_EXP / "resultado.json", encoding="utf-8"))

EXPERIMENTO  = CFG['experimento']
DATASET_FE   = CFG['dataset_fe']
TARGET       = CFG['target']
TARGET_KIND  = CFG['target_kind']
METODO       = CFG['metodo_normalizacion']
H            = CFG['horizonte']
FEATURES     = CFG['features']
CAT_FEATURES = CFG['cat_features']
HIPER        = CFG['hiperparametros']

print(f"Experimento : {EXPERIMENTO}")
print(f"Dataset FE  : {DATASET_FE}")
print(f"Target      : {TARGET}  (kind={TARGET_KIND}, norm={METODO}, horizonte={H})")
print(f"Features    : {len(FEATURES)}   categoricas: {CAT_FEATURES}")
print(f"WAPE val    : {CFG.get('wape_val')}")
print(f"WAPE test   : {CFG.get('wape_test')}")
print(f"Leakage     : {CFG.get('leakage')}")
print(f"\nHiperparametros:")
for k, v in HIPER.items():
    print(f"   {k:22s} {v}")

## 2 — Datos

In [ ]:
path_in = RUTA_FE / DATASET_FE
if not path_in.exists():
    raise FileNotFoundError(f"No existe {path_in}. Corre 02_FE con esas palancas.")

df = pl.read_parquet(path_in)
print(f"Dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas")

# Misma separacion que 03: target nulo = filas de inferencia (todavia no se conoce t+H)
df_sup   = df.filter(pl.col(TARGET).is_not_null())
df_infer = df.filter(pl.col(TARGET).is_null())

meses_sup = sorted(df_sup['periodo'].unique().to_list())
meses_inf = sorted(df_infer['periodo'].unique().to_list())
print(f"\nSupervisado : {df_sup.height:,} filas   meses {meses_sup[0]} -> {meses_sup[-1]}")
print(f"Inferencia  : {df_infer.height:,} filas   meses {meses_inf}")

# A diferencia de 03, ACA SE ENTRENA CON TODO lo supervisado: los meses que 03
# habia reservado para val y test ya cumplieron su funcion.
usados_en_03 = set(CFG['meses_train']) | set(CFG['meses_val']) | set(CFG['meses_test'])
extra = sorted(set(meses_sup) - set(CFG['meses_train']))
print(f"\n03 entreno con {len(CFG['meses_train'])} meses; aca se usan {len(meses_sup)}.")
print(f"Meses que 03 no habia usado para entrenar: {extra}")

In [ ]:
IDS = [c for c in ['product_id', 'customer_id', 'Agrupacion_ID'] if c in df.columns]
COLS_CTX = [c for c in ['B0', 'B1', 'tn0_norm'] + IDS if c in df.columns]

faltan = [c for c in FEATURES if c not in df.columns]
if faltan:
    raise ValueError(f"El dataset no tiene {len(faltan)} features del experimento: {faltan[:10]}")

_cols = sorted(set(FEATURES + COLS_CTX + ['periodo'] + [TARGET]))
train_pd = df_sup.select(_cols).to_pandas()
infer_pd = df_infer.select([c for c in _cols if c in df_infer.columns]).to_pandas()

# Las categoricas de inferencia deben compartir el mismo diccionario que las de train
for c in CAT_FEATURES:
    train_pd[c] = train_pd[c].astype('category')
    if c in infer_pd.columns:
        infer_pd[c] = infer_pd[c].astype('category').cat.set_categories(
            train_pd[c].cat.categories)

print(f"train: {train_pd.shape}   inferencia: {infer_pd.shape}")
if infer_pd.empty:
    raise RuntimeError("No hay filas de inferencia: sin ellas no se puede armar la entrega.")

## 3 — Entrenamiento del ensemble

Cada semilla cambia el submuestreo de filas y de columnas, así que los modelos
cometen errores distintos. Promediarlos cancela parte de esa varianza.

`deterministic=True` para que dos corridas del notebook den el mismo submit — sin
eso, LightGBM multihilo no es reproducible y no se puede rastrear qué generó cada
entrega.

In [ ]:
def params_finales(semilla):
    p = dict(HIPER)
    p.update({
        'objective':      CFG['objective_lgbm'],
        'metric':         'mae',
        'verbosity':      -1,
        'boosting_type':  'gbdt',
        'n_jobs':         -1,
        'subsample_freq': 1,
        'seed':           semilla,
        'deterministic':  True,
        'force_row_wise': True,
    })
    return p


modelos = []
for i, sem in enumerate(PARAM['semillas_ensemble'], 1):
    m = lgb.LGBMRegressor(**params_finales(sem))
    m.fit(train_pd[FEATURES], train_pd[TARGET], categorical_feature=CAT_FEATURES)
    modelos.append(m)
    print(f"[{i}/{len(PARAM['semillas_ensemble'])}] semilla {sem} entrenada")

print(f"\n{len(modelos)} modelo(s) en el ensemble, {len(train_pd):,} filas cada uno")

## 4 — Predicción y reconstrucción a toneladas

Sea cual sea la variable respuesta, la predicción del modelo se lleva a toneladas
con los `B0`/`B1` de cada fila. Es el inverso exacto de la normalización de `02_FE`.

In [ ]:
def reconstruir_nivel(pred, ctx: pd.DataFrame) -> np.ndarray:
    """Pasa la prediccion a toneladas segun la variable respuesta del experimento."""
    pred = np.asarray(pred, dtype=np.float64)
    if TARGET_KIND == 'nivel':
        return pred
    if TARGET_KIND == 'delta':
        # clase_tn_delta = clase_tn_norm - tn0_norm
        pred = pred + ctx['tn0_norm'].to_numpy(dtype=np.float64)
    B0 = ctx['B0'].to_numpy(dtype=np.float64)
    B1 = ctx['B1'].to_numpy(dtype=np.float64)
    if METODO == 'recta':
        return pred + (B0 + B1 * (-float(H)))
    B1s = np.where((B1 == 0) | ~np.isfinite(B1), 1.0, B1)
    return pred * B1s + B0


# Chequeo de sanidad: reconstruir el TARGET REAL sobre train debe devolver clase_tn.
if 'clase_tn' in df_sup.columns:
    _m = train_pd.head(20_000)
    _rec = reconstruir_nivel(_m[TARGET].to_numpy(), _m)
    _real = df_sup.select('clase_tn').to_pandas()['clase_tn'].to_numpy()[:len(_m)]
    _err = float(np.nanmax(np.abs(_rec - _real)))
    print(f"Round-trip de reconstruccion: error maximo = {_err:.10f}")
    if _err > 1e-6:
        raise RuntimeError(f"La reconstruccion no cierra (error {_err}). Revisa METODO={METODO!r}.")
    print("Reconstruccion validada.")

# Promedio del ensemble sobre la escala del target, y despues a toneladas
preds = np.column_stack([m.predict(infer_pd[FEATURES]) for m in modelos])
y_pred = reconstruir_nivel(preds.mean(axis=1), infer_pd)
y_pred = np.maximum(y_pred, PARAM['clip_min'])

def sumar_meses(p, k):
    m = (p // 100) * 12 + (p % 100) - 1 + k
    return (m // 12) * 100 + (m % 12) + 1

pred = infer_pd[['periodo'] + IDS].copy()
pred['tn_pred'] = y_pred
pred['periodo_objetivo'] = pred['periodo'].map(lambda p: sumar_meses(int(p), H))

print(f"\nPredicciones: {len(pred):,} filas")
print(pred.groupby(['periodo', 'periodo_objetivo']).size().rename('filas').reset_index().to_string(index=False))
print(f"\ntn_pred   min {y_pred.min():.3f}   media {y_pred.mean():.3f}   max {y_pred.max():.3f}")
print(f"predicciones en 0: {(y_pred == 0).sum():,}")

## 5 — Armado de la entrega

Kaggle mide a nivel `product_id`. Con granularidad producto-cliente hay que **sumar
las predicciones de todos los clientes** de cada producto — es la misma agregación
con la que `03` midió el WAPE, así que el número de la entrega es comparable con el
del leaderboard.

Los productos de la lista oficial que no tengan predicción van con **0**. Que sean
muchos es una señal de alarma, no algo normal: significa que el pipe perdió
productos en el camino.

In [ ]:
OBJ = PARAM['periodo_objetivo']
pred_obj = pred[pred['periodo_objetivo'] == OBJ]
if pred_obj.empty:
    raise RuntimeError(
        f"No hay predicciones para {OBJ}. Objetivos disponibles: "
        f"{sorted(pred['periodo_objetivo'].unique())}")

por_producto = (pred_obj.groupby('product_id', as_index=False)['tn_pred']
                        .sum().rename(columns={'tn_pred': 'tn'}))
print(f"Mes objetivo {OBJ}: {len(pred_obj):,} filas -> {len(por_producto)} productos")

path_apredecir = RUTA_RAW / "product_id_apredecir201912.txt"
if not path_apredecir.exists():
    raise FileNotFoundError(f"Falta {path_apredecir}")
oficiales = pl.read_csv(path_apredecir, separator="\t").to_pandas()
print(f"Productos en la lista oficial: {len(oficiales)}")

submit = (oficiales[['product_id']]
          .merge(por_producto, on='product_id', how='left'))
sin_pred = submit['tn'].isna().sum()
submit['tn'] = submit['tn'].fillna(0.0)
submit = submit.sort_values('product_id').reset_index(drop=True)

print(f"\nSubmit: {len(submit)} filas")
print(f"Productos SIN prediccion (van en 0): {sin_pred}")
if sin_pred > 0:
    faltantes = submit.loc[submit['tn'] == 0, 'product_id'].tolist()
    print(f"   {faltantes[:20]}{' ...' if len(faltantes) > 20 else ''}")
    if sin_pred > len(oficiales) * 0.05:
        print(f"   ATENCION: es mas del 5% de la lista. Revisa si 01_Preprocesamiento")
        print(f"   corrio con filter_target_products_only=True y sin muestreo.")
print(f"\ntn   min {submit['tn'].min():.3f}   media {submit['tn'].mean():.3f}   "
      f"max {submit['tn'].max():.3f}   suma {submit['tn'].sum():,.1f}")
print(submit.head(10).to_string(index=False))

In [ ]:
path_submit = DIR_EXP / f"submission_{OBJ}.csv"
submit.to_csv(path_submit, index=False)
print(f"Guardado: {path_submit}")

# Copia con nombre fijo, para encontrar siempre la ultima
shutil.copy(path_submit, RUTA_EXP / "submission_ultima.csv")
print(f"Copia    : {RUTA_EXP/'submission_ultima.csv'}")
print()
print(open(path_submit).read()[:300])

## 6 — Submit a Kaggle

Necesita `~/.kaggle/kaggle.json` con permisos `600`. La celda lo busca en el home y,
si no está, lo copia del bucket.

In [ ]:
kaggle_dst = Path.home() / ".kaggle" / "kaggle.json"
kaggle_dst.parent.mkdir(parents=True, exist_ok=True)

if kaggle_dst.exists():
    kaggle_dst.chmod(0o600)
    print(f"Kaggle auth OK: {kaggle_dst}")
else:
    for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
        if cand.exists():
            shutil.copy(cand, kaggle_dst)
            kaggle_dst.chmod(0o600)
            print(f"Kaggle auth copiada de {cand}")
            break
    else:
        print("kaggle.json NO encontrado.")
        print("Bajalo de kaggle.com -> Settings -> API -> Create New Token")
        print(f"y dejalo en {kaggle_dst} o en {BUCKET}/kaggle.json")

In [ ]:
if not PARAM['submit']:
    print("PARAM['submit'] = False -> no se sube nada. El CSV ya esta generado.")
elif not kaggle_dst.exists():
    print("Sin credenciales de Kaggle: no se sube. El CSV ya esta generado.")
else:
    msg = PARAM['mensaje_submit'] or (
        f"{EXPERIMENTO[:80]} | wape_test={CFG.get('wape_test')} | "
        f"ensemble={len(modelos)} semillas")
    res = subprocess.run(
        ['kaggle', 'competitions', 'submit',
         '-c', PARAM['kaggle_competition'],
         '-f', str(path_submit),
         '-m', msg],
        capture_output=True, text=True)
    print(f"mensaje : {msg}")
    print(f"returncode: {res.returncode}")
    print(f"stdout: {res.stdout}")
    if res.stderr:
        print(f"stderr: {res.stderr}")
    if res.returncode == 0:
        print("\nSubmit enviado. Verificalo con la celda de abajo.")

In [ ]:
# Ultimos submits de la competencia
res = subprocess.run(['kaggle', 'competitions', 'submissions',
                      '-c', PARAM['kaggle_competition']],
                     capture_output=True, text=True)
print(res.stdout or res.stderr)

## 7 — Registro de la entrega

Deja constancia de qué modelo generó qué submit, para poder reconstruirlo después.

In [ ]:
registro = {
    'experimento':        EXPERIMENTO,
    'dataset_fe':         DATASET_FE,
    'periodo_objetivo':   OBJ,
    'archivo':            str(path_submit),
    'n_productos':        int(len(submit)),
    'n_sin_prediccion':   int(sin_pred),
    'tn_total_predicho':  float(submit['tn'].sum()),
    'semillas_ensemble':  PARAM['semillas_ensemble'],
    'meses_entrenamiento': [int(m) for m in meses_sup],
    'wape_val_en_03':     CFG.get('wape_val'),
    'wape_test_en_03':    CFG.get('wape_test'),
    'hiperparametros':    HIPER,
}
with open(DIR_EXP / f"submit_{OBJ}.json", 'w', encoding='utf-8') as f:
    json.dump(registro, f, indent=2, ensure_ascii=False, default=str)

print(f"Registro: {DIR_EXP/f'submit_{OBJ}.json'}")
print(json.dumps({k: v for k, v in registro.items() if k != 'hiperparametros'},
                 indent=2, ensure_ascii=False, default=str))